# Notebook 2: Analyze Visium Fluorescence Data

**Source:** [Squidpy Tutorial — Visium Fluorescence](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_visium_fluo.html)

This tutorial shows how to apply Squidpy's **image analysis** features for Visium fluorescence data. The dataset is a cropped coronal section of the mouse brain with three fluorescence channels:
- **DAPI** — DNA staining (all nuclei)
- **anti-NEUN** — neuronal marker
- **anti-GFAP** — glial cell marker

## Topics Covered
1. Loading pre-processed Visium fluorescence data
2. Image segmentation with watershed algorithm
3. Segmentation feature extraction (cell counts, channel intensities)
4. Multi-scale image feature extraction (summary, histogram, texture)
5. Leiden clustering in image feature space

In [ ]:
!pip install scanpy squidpy leidenalg python-igraph seaborn openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of spatialdata to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of spatialdata to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of dask[array,dataframe] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of ome-zarr to determine which version is compatible with other requirements. This could take a 

## 0. Import Libraries

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt

import anndata as ad
import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

squidpy==1.8.1


## 1. Load Data

Squidpy provides pre-processed datasets. Here we load:
- `adata`: AnnData with count matrix and cluster annotations
- `img`: ImageContainer with the fluorescence tissue image

In [ ]:
# Load pre-processed Visium fluorescence dataset (mouse brain crop)
img = sq.datasets.visium_fluo_image_crop()
adata = sq.datasets.visium_fluo_adata_crop()

print(adata)
print("\nCluster labels:", adata.obs["cluster"].cat.categories.tolist())

INFO     Downloading visium_fluo_image_crop.tiff from                                                              
         https://exampledata.scverse.org/squidpy/figshare/visium_fluo_image_crop.tiff                              


  0%|                                               | 0.00/317M [00:00<?, ?B/s]

INFO     Downloading visium_fluo_adata_crop.h5ad from                                                              
         https://exampledata.scverse.org/squidpy/figshare/visium_fluo_adata_crop.h5ad                              


  0%|                                              | 0.00/68.7M [00:00<?, ?B/s]

AnnData object with n_obs × n_vars = 704 × 16562
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_MT', 'log1p_total_counts_MT', 'pct_counts_MT', 'n_counts', 'leiden', 'cluster'
    var: 'gene_ids', 'feature_types', 'genome', 'MT', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cluster_colors', 'hvg', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'spatial', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

Cluster labels: ['Cortex_1', 'Cortex_2', 'Cortex_3', 'Cortex_4', 'Dentate_gyrus', 'Fiber_tracts', 'Hippocampus', 'Hypothalamus_1', 'Lateral

In [ ]:
# Visualize cluster annotation in spatial context
sq.pl.spatial_scatter(adata, color="cluster")

<Figure size 640x480 with 1 Axes>

In [ ]:
# Visualize fluorescence channels separately
# Channel 0: DAPI, Channel 1: anti-NEUN, Channel 2: anti-GFAP
img.show(channelwise=True)

<Figure size 800x800 with 3 Axes>

## 2. Image Segmentation

Before extracting segmentation features, we need to **segment cells** in the fluorescence image.  

Steps:
1. **Smooth** the DAPI channel with a Gaussian filter
2. **Segment** cells using the **watershed** algorithm
3. Visualize segmented cells alongside the raw DAPI image

In [ ]:
# Step 1: Smooth the image
sq.im.process(
    img=img,
    layer="image",
    method="smooth"
)

# Step 2: Segment using watershed on DAPI channel (channel_id=0)
sq.im.segment(
    img=img,
    layer="image_smooth",
    method="watershed",
    channel=0,
    chunks=1000
)

In [ ]:
# Step 3: Visualize segmentation result on a crop
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

img_crop = img.crop_corner(2000, 2000, size=500)

img_crop.show(layer="image", channel=0, ax=ax[0])
ax[0].set_title("Raw DAPI channel")

img_crop.show(layer="segmented_watershed", channel=0, ax=ax[1])
ax[1].set_title("Watershed segmentation")

plt.tight_layout()

plt.show()

print("Segmentation stored in: img['segmented_watershed']")
print("Each unique integer = one segmented cell/nucleus")

<Figure size 1200x500 with 2 Axes>

Segmentation stored in: img['segmented_watershed']
Each unique integer = one segmented cell/nucleus


## 3. Segmentation Features

Using the segmentation mask, we extract per-spot features:
- **`segmentation_label`**: number of cells per Visium spot
- **`segmentation_ch-X_mean_intensity_mean`**: mean fluorescence intensity of each channel

This approximates **cell density** and **cell type composition** per spot.

In [ ]:
# Define segmentation layer
features_kwargs = {"segmentation": {"label_layer": "segmented_watershed"}}

# Calculate segmentation features
sq.im.calculate_image_features(
    adata,
    img,
    features="segmentation",
    layer="image",
    key_added="features_segmentation",
    n_jobs=1,
    features_kwargs=features_kwargs
)

  0%|          | 0/704 [00:00<?, ?/s]

In [ ]:
# Plot segmentation features alongside gene-space clusters
sq.pl.spatial_scatter(
    sq.pl.extract(adata, "features_segmentation"),
    color=[
        "segmentation_label",           # cell count per spot
        "cluster",                       # gene-space cluster
        "segmentation_ch-0_mean_intensity_mean",  # DAPI intensity
        "segmentation_ch-1_mean_intensity_mean",  # anti-NEUN intensity
    ],
    frameon=False,
    ncols=2
)

<Figure size 1455.6x960 with 7 Axes>

**Key Observations:**
- Cell-rich **pyramidal layer of the Hippocampus** shows higher cell counts than surrounding regions
- *Cortex_1* and *Cortex_3* clusters show higher **anti-NEUN** (neuronal) signal
- *Fiber_tracts* and *lateral ventricles* show higher **anti-GFAP** (glial) signal

## 4. Multi-Scale Image Feature Extraction

We extract **summary**, **histogram**, and **texture** features at different spatial scales. This creates a rich feature matrix per spot that captures tissue morphology at multiple resolutions.

In [ ]:
# Define feature extraction configurations
params = {
    # Original resolution, tissue under spot only
    "features_orig": {
        "features": ["summary", "texture", "histogram"],
        "scale": 1.0,
        "mask_circle": True,
    },
    # More context at original resolution
    "features_context": {
        "features": ["summary", "histogram"],
        "scale": 1.0
    },
    # More context at lower resolution
    "features_lowres": {
        "features": ["summary", "histogram"],
        "scale": 0.25
    },
}

for feature_name, cur_params in params.items():
    sq.im.calculate_image_features(
        adata, img,
        layer="image",
        key_added=feature_name,
        n_jobs=1,
        **cur_params
    )
    print(f"Computed: {feature_name}")

  0%|          | 0/704 [00:00<?, ?/s]

Computed: features_orig


  0%|          | 0/704 [00:00<?, ?/s]

Computed: features_context


  0%|          | 0/704 [00:00<?, ?/s]

Computed: features_lowres


In [ ]:
# Concatenate all feature matrices
adata.obsm["features"] = pd.concat(
    [adata.obsm[f] for f in params.keys()],
    axis="columns"
)

# Ensure unique column names
adata.obsm["features"].columns = ad.utils.make_index_unique(
    adata.obsm["features"].columns
)

print(f"Combined feature matrix shape: {adata.obsm['features'].shape}")

Combined feature matrix shape: (704, 195)


## 5. Leiden Clustering in Image Feature Space

We cluster spots based on their **image features** and compare to gene-expression clusters.

In [ ]:
def cluster_features(features: pd.DataFrame, like=None) -> pd.Series:
    """Compute Leiden clustering of image features.

    Parameters
    ----------
    features
        DataFrame of image features (spots × features).
    like
        Substring to filter column names (e.g. 'summary', 'histogram').

    Returns
    -------
    pd.Series
        Leiden cluster labels.
    """
    if like is not None:
        features = features.filter(like=like)

    # Create temporary AnnData for clustering
    tmp_adata = ad.AnnData(features)
    sc.pp.scale(tmp_adata)        # scale before PCA — features are not normalized
    sc.pp.pca(tmp_adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(tmp_adata)
    sc.tl.leiden(tmp_adata)

    return tmp_adata.obs["leiden"]

In [ ]:
# Cluster based on different feature types
adata.obs["features_summary_cluster"] = cluster_features(
    adata.obsm["features"], like="summary"
)
adata.obs["features_histogram_cluster"] = cluster_features(
    adata.obsm["features"], like="histogram"
)
adata.obs["features_texture_cluster"] = cluster_features(
    adata.obsm["features"], like="texture"
)

/tmp/ipykernel_4999/3548582851.py:24: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(tmp_adata)
/tmp/ipykernel_4999/3548582851.py:24: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(tmp_adata)
/tmp/ipykernel_4999/3548582851.py:24: FutureWarning: The `igra

In [ ]:
# Compare image feature clusters vs. gene-expression clusters
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sq.pl.spatial_scatter(
    adata,
    color=[
        "features_summary_cluster",
        "features_histogram_cluster",
        "features_texture_cluster",
        "cluster",
    ],
    ncols=2
)

<Figure size 1425.6x1280 with 4 Axes>

## Summary

**Key Takeaways:**
- Image-based clusters are **spatially coherent**, just like gene-based clusters
- Fluorescence features provide **complementary information** — the Hippocampus (one gene cluster) is subdivided into multiple distinct morphological units by image features
- Different feature extractors (summary, histogram, texture) capture different aspects of tissue morphology
- Cell segmentation enables per-spot **cell counting** and **cell type estimation** from channel intensities
